# tabtransformer

## 1.import libraries

In [9]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

## 2.load data

In [10]:
from preprocessing_22 import load_data_scaled

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    feature_names
) = load_data_scaled()

## 3.Convert to PyTorch Tensors

In [11]:
X_train = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

X_val = torch.tensor(
    X_val.values,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test.values,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

y_val = torch.tensor(
    y_val.values,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test.values,
    dtype=torch.float32
)

## 4.Create DataLoaders

In [12]:
batch_size = 1024

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=batch_size
)

## 5.Simpler TabTransformer-style Model

In [13]:
class TabTransformerModel(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.embedding = nn.Linear(
            input_dim,
            64
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        x = self.embedding(x)

        x = x.unsqueeze(1)

        x = self.transformer(x)

        x = x.squeeze(1)

        x = self.classifier(x)

        return x

## 6.Initialize Model

In [14]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = TabTransformerModel(
    input_dim=X_train.shape[1]
).to(device)

## 7.Loss and Optimizer

In [15]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## 8.tain model

In [16]:
epochs = 20

for epoch in range(epochs):

    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze()

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {train_loss:.4f}"
    )

Epoch 1/20, Loss: 217.6709
Epoch 2/20, Loss: 205.9847
Epoch 3/20, Loss: 203.4003
Epoch 4/20, Loss: 201.7869
Epoch 5/20, Loss: 200.5386
Epoch 6/20, Loss: 199.5577
Epoch 7/20, Loss: 198.7817
Epoch 8/20, Loss: 198.3358
Epoch 9/20, Loss: 197.8402
Epoch 10/20, Loss: 197.1421
Epoch 11/20, Loss: 196.6317
Epoch 12/20, Loss: 196.2566
Epoch 13/20, Loss: 195.8141
Epoch 14/20, Loss: 195.4024
Epoch 15/20, Loss: 194.8449
Epoch 16/20, Loss: 194.2445
Epoch 17/20, Loss: 193.8324
Epoch 18/20, Loss: 193.2087
Epoch 19/20, Loss: 192.9884
Epoch 20/20, Loss: 192.5187


## 9.predictions

In [17]:
model.eval()

with torch.no_grad():

    logits = model(
        X_test.to(device)
    ).squeeze()

    y_prob = torch.sigmoid(
        logits
    ).cpu().numpy()

    y_pred = (
        y_prob > 0.5
    ).astype(int)

## 10.matrics

In [18]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    y_prob
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {auc:.4f}")

Accuracy : 0.8418
Precision: 0.7895
Recall   : 0.7320
F1 Score : 0.7597
ROC AUC  : 0.9104


In [19]:
torch.save(
    model.state_dict(),
    "../models/higgs_tabtransformer.pth"
)